# ZeroID Quickstart — Agent Identity Platform

**End-to-end identity, authentication, and delegation for autonomous agents.**

### The Problem

As AI agents become autonomous — calling APIs, accessing databases, spawning sub-agents — a critical question emerges: **"Who is this agent, what is it allowed to do, and who authorized it?"**

Traditional auth systems were built for humans clicking buttons. They don't answer:
- Which agent made this API call? (not just "some service account")
- Was it acting on its own, or delegated by another agent?
- What's its trust level? Is it a verified first-party agent or an unknown third-party?
- Can we revoke its access in real-time if it behaves anomalously?

### How ZeroID Helps

ZeroID gives every agent a **cryptographic identity** with:
- A stable **WIMSE/SPIFFE URI** as its globally unique identifier
- **Short-lived JWTs** issued via standard OAuth2 flows (no long-lived API keys)
- **Scoped delegation** — orchestrators can delegate authority to sub-agents with downscoped permissions
- **Trust levels** that advance through attestation (unverified → verified → first_party)
- **Real-time revocation** via Continuous Access Evaluation (CAE) signals
- **Full audit trail** — every token records who issued it, who delegated, and what scopes were granted

### This Notebook

We'll walk through the full lifecycle:

1. **Setup & Connection** — install, connect, health check
2. **Register Agent Identities** — orchestrator + tool agent with ECDSA keys
3. **OAuth2 Client Credentials** — get a token, decode JWT claims
4. **Agent-to-Agent Delegation (RFC 8693)** — orchestrator delegates to tool agent
5. **Token Introspection & Revocation** — inspect and revoke tokens
6. **Credential Policies** — govern token issuance with policies
7. **CAE Signals** — real-time revocation on anomalous behavior
8. **Downstream Authorization** — how ZeroID tokens enable fine-grained access control

---

## Prerequisites

### 1. Install the Highflame SDK

```bash
pip install highflame cryptography PyJWT
```

### 2. Start ZeroID locally

Clone the ZeroID repo and start the service with Docker Compose:

```bash
cd zeroid
make setup-keys           # generate ECDSA P-256 signing keys
docker compose up -d      # starts PostgreSQL + ZeroID on port 8899
```

ZeroID exposes two logical surfaces on the same port:
- **Public endpoints** (`/oauth2/*`, `/health`, `/.well-known/*`) — no auth required
- **Admin endpoints** (`/api/v1/*`) — tenant context via `X-Account-ID` + `X-Project-ID` headers

In [ ]:
#!pip install -q highflame cryptography PyJWT

---

## 1. Setup & Connection

The `ZeroIDClient` wraps both the admin API (identity management, policies, signals)
and the public OAuth2 endpoints (token, introspect, revoke) in a single client.

In [ ]:
import json
import time
import base64
from datetime import datetime, timezone

from highflame.zeroid import ZeroIDClient


def pp(obj):
    """Pretty-print a dict or object as indented JSON."""
    if hasattr(obj, "__dict__"):
        obj = obj.__dict__
    print(json.dumps(obj, indent=2, default=str))


def decode_jwt_claims(token: str) -> dict:
    """Decode JWT payload without verification (for display only)."""
    payload = token.split(".")[1]
    # Add padding
    payload += "=" * (4 - len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


def _table(headers, rows):
    """Print a formatted table."""
    widths = [len(h) for h in headers]
    str_rows = [[str(c) for c in row] for row in rows]
    for row in str_rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(cell))
    sep = "  "
    fmt = sep.join(f"{{:<{w}}}" for w in widths)
    print(fmt.format(*headers))
    print(sep.join("\u2500" * w for w in widths))
    for row in str_rows:
        print(fmt.format(*row))

In [ ]:
# Connect to local ZeroID instance.
# account_id and project_id are auto-generated if not provided —
# every resource is scoped to this tenant.
client = ZeroIDClient(base_url="http://localhost:8899")

print(f"ZeroID client configured:")
print(f"  Base URL:   {client._base_url}")
print(f"  Account:    {client.account_id}")
print(f"  Project:    {client.project_id}")
print()
print("(Account and project auto-generated for this session. Pass them explicitly in production.)")

### Health Check

Verify ZeroID is running and the database is reachable.

In [ ]:
health = client.health()

print(f"Status:    {health.status}")
print(f"Service:   {health.service}")
print(f"Uptime:    {health.uptime_ms}ms")
print(f"Timestamp: {health.timestamp}")

### OAuth2 Server Discovery

ZeroID publishes its capabilities at `/.well-known/oauth-authorization-server` (RFC 8414).

In [ ]:
# oauth_metadata() is not in the SDK — fetch the well-known endpoint directly.
import httpx

metadata = httpx.get(f"{client._base_url}/.well-known/oauth-authorization-server").json()

print(f"Issuer:           {metadata['issuer']}")
print(f"Token endpoint:   {metadata['token_endpoint']}")
print(f"JWKS URI:         {metadata['jwks_uri']}")
print(f"Grant types:      {metadata['grant_types_supported']}")
print(f"Signing algs:     {metadata['token_endpoint_auth_signing_alg_values_supported']}")

---

## 2. Register Agent Identities

### Why Agent Identity Matters

Without identity, agents are invisible. You can't distinguish a billing agent from a data pipeline,
can't audit which agent accessed customer records, and can't revoke one agent without killing them all.

ZeroID solves this by giving every agent a **registered identity** — just like humans have user accounts,
agents get identity records with classification, trust levels, and capabilities.

### What Each Identity Gets

- A stable **WIMSE URI** (SPIFFE-compatible): `spiffe://zeroid.dev/{account}/{project}/{type}/{external_id}`
- A **trust level**: `unverified` → `verified_third_party` → `first_party` (advances through attestation)
- A **type** + **sub-type** classification (e.g., `agent/orchestrator`, `agent/tool_agent`)
- Optional **ECDSA P-256 public key** for secretless authentication (jwt_bearer grant)

### Identity Type Taxonomy

| Identity Type | Sub-Types | When To Use |
|--------------|-----------|-------------|
| `agent` | `orchestrator`, `autonomous`, `tool_agent`, `human_proxy`, `evaluator` | AI agents that reason, plan, and act |
| `application` | `chatbot`, `assistant`, `api_service`, `code_agent`, `custom` | User-facing apps that call your APIs |
| `mcp_server` | (none) | MCP tool servers (Claude, Cursor, etc.) |
| `service` | (none) | Internal platform services (monitoring, ETL) |

### Register an Orchestrator Agent

The orchestrator coordinates sub-agents and workflows. It authenticates via OAuth2 client credentials.

In [ ]:
from highflame.zeroid import ConflictError

try:
    orchestrator = client.identities.create(
        external_id="billing-orchestrator",
        name="Billing Orchestrator Agent",
        identity_type="agent",
        sub_type="orchestrator",
        trust_level="first_party",
        owner_user_id="user-yash",
        allowed_scopes=["billing:read", "billing:write", "data:read"],
        framework="langchain",
        version="0.3.1",
        description="Orchestrates billing workflows across sub-agents",
    )
    print("Identity registered (new)")
except ConflictError:
    # Already exists — fetch it by listing and filtering
    identities = client.identities.list()
    orchestrator = next(i for i in identities if i.external_id == "billing-orchestrator")
    print("Identity already exists (reusing)")

print(f"  ID:            {orchestrator.id}")
print(f"  External ID:   {orchestrator.external_id}")
print(f"  Type:          {orchestrator.identity_type}/{orchestrator.sub_type}")
print(f"  Trust Level:   {orchestrator.trust_level}")
print(f"  Status:        {orchestrator.status}")
print(f"  WIMSE URI:     {orchestrator.wimse_uri}")
print(f"  Scopes:        {orchestrator.allowed_scopes}")

### WIMSE URI Format

Every identity gets a stable, globally unique URI following the WIMSE/SPIFFE convention.
This URI becomes the `sub` claim in all issued JWTs.

```
spiffe://zeroid.dev/{account_id}/{project_id}/{identity_type}/{external_id}
```

Example: `spiffe://zeroid.dev/acct-demo-001/proj-demo-001/agent/billing-orchestrator`

In [ ]:
wimse_uri = orchestrator.wimse_uri
print(f"WIMSE URI: {wimse_uri}")

# Parse the URI components
parts = wimse_uri.replace("spiffe://", "").split("/")
_table(
    ["Component", "Value"],
    [
        ["Domain", parts[0]],
        ["Account ID", parts[1]],
        ["Project ID", parts[2]],
        ["Identity Type", parts[3]],
        ["External ID", parts[4]],
    ],
)

### Register a Tool Agent with ECDSA P-256 Public Key

**When to use key-based identity (jwt_bearer):** When the agent manages its own keys and
you don't want shared secrets. The agent holds its private key securely and proves its
identity by signing a JWT assertion — no secret ever transmitted over the network.

This is the preferred pattern for:
- **Third-party agents** you don't control (they bring their own keys)
- **High-security environments** where shared secrets are not acceptable
- **Agent-to-agent delegation** where the sub-agent needs to prove identity independently

In [ ]:
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import serialization

# Generate the tool agent's ECDSA P-256 key pair
# In production, this key would be generated and stored securely by the agent.
tool_agent_private_key = ec.generate_private_key(ec.SECP256R1())
tool_agent_public_key = tool_agent_private_key.public_key()

# Export public key as PEM for registration
public_key_pem = tool_agent_public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
).decode()

print("Tool agent ECDSA P-256 public key (PEM):")
print(public_key_pem)

In [ ]:
try:
    tool_agent = client.identities.create(
        external_id="data-fetcher",
        name="Data Fetcher Tool Agent",
        identity_type="agent",
        sub_type="tool_agent",
        trust_level="first_party",
        owner_user_id="user-yash",
        allowed_scopes=["data:read"],
        public_key_pem=public_key_pem,
        framework="custom",
        version="1.0.0",
        description="Fetches data from internal databases",
    )
    print("Identity registered (new)")
except ConflictError:
    # Already exists — fetch it by listing and filtering
    identities = client.identities.list()
    tool_agent = next(i for i in identities if i.external_id == "data-fetcher")
    # Update with fresh public key (regenerated above)
    tool_agent = client.identities.update(tool_agent.id, public_key_pem=public_key_pem)
    print("Identity already exists (reusing, key updated)")

print(f"  ID:            {tool_agent.id}")
print(f"  External ID:   {tool_agent.external_id}")
print(f"  Type:          {tool_agent.identity_type}/{tool_agent.sub_type}")
print(f"  WIMSE URI:     {tool_agent.wimse_uri}")
print(f"  Has public key: {bool(tool_agent.public_key_pem)}")

### Verify Identities

Confirm both identities are registered and active.

In [ ]:
# Verify both identities
_table(
    ["Agent", "Status", "Trust Level", "WIMSE URI"],
    [
        [orchestrator.name, orchestrator.status, orchestrator.trust_level, orchestrator.wimse_uri],
        [tool_agent.name, tool_agent.status, tool_agent.trust_level, tool_agent.wimse_uri],
    ],
)

---

## 3. OAuth2 Client Credentials Flow

### When To Use This

This is the standard machine-to-machine flow — your platform provisions an OAuth client for an agent,
and the agent uses `client_id` + `client_secret` to get short-lived tokens. Use this when:

- **Your platform manages the agent's secrets** (you control deployment)
- **The agent runs in your infrastructure** (not a third-party agent)
- **You want the simplest possible auth flow** (no crypto keys needed)

The token expires in 1 hour by default — no long-lived API keys sitting in config files.

```
Agent                         ZeroID
  |                              |
  |-- POST /oauth2/token ------->|
  |   grant_type: client_creds   |
  |   client_id + client_secret  |
  |                              |
  |<---- { access_token: JWT } --|
  |       (1hr TTL, ES256)       |
```

### Create OAuth Client for the Orchestrator

In [ ]:
try:
    oauth_result = client.oauth_clients.create(
        name="billing-orchestrator-client",
        external_id="billing-orchestrator",
        grant_types=["client_credentials"],
        scopes=["billing:read", "billing:write", "data:read"],
    )
    oauth_client = oauth_result.client
    client_secret = oauth_result.client_secret
    print("OAuth client created (new)")
except ConflictError:
    # Already exists — find it and rotate the secret to get a fresh one
    clients = client.oauth_clients.list()
    existing = next(c for c in clients if c.client_id == "billing-orchestrator")
    rotated = client.oauth_clients.rotate_secret(existing.id)
    oauth_client = rotated.client
    client_secret = rotated.client_secret
    print("OAuth client already exists (secret rotated)")

print(f"  Client ID:     {oauth_client.client_id}")
print(f"  Client Secret: {client_secret[:20]}...  (save this!)")
print(f"  Grant Types:   {oauth_client.grant_types}")
print(f"  Scopes:        {oauth_client.scopes}")
print(f"  Identity ID:   {oauth_client.identity_id}")

### Get a Token via Client Credentials

In [ ]:
token_response = client.tokens.issue(
    grant_type="client_credentials",
    client_id=oauth_client.client_id,
    client_secret=client_secret,
    scope="billing:read data:read",
)

orchestrator_token = token_response.access_token

print(f"Token issued:")
print(f"  Token Type:  {token_response.token_type}")
print(f"  Expires In:  {token_response.expires_in}s")
print(f"  Scope:       {token_response.scope}")
print(f"  JTI:         {token_response.jti}")
print(f"  Token:       {orchestrator_token[:60]}...")

### Decode JWT Claims

ZeroID JWTs carry rich identity context. Downstream services can make authorization
decisions using these claims without calling back to ZeroID.

In [ ]:
claims = decode_jwt_claims(orchestrator_token)

print("JWT Claims:")
print(f"  iss (issuer):        {claims.get('iss')}")
print(f"  sub (subject):       {claims.get('sub')}")
print(f"  jti (token ID):      {claims.get('jti')}")
print(f"  account_id:          {claims.get('account_id')}")
print(f"  project_id:          {claims.get('project_id')}")
print(f"  identity_type:       {claims.get('identity_type')}")
print(f"  sub_type:            {claims.get('sub_type')}")
print(f"  trust_level:         {claims.get('trust_level')}")
print(f"  scopes:              {claims.get('scopes')}")
print(f"  grant_type:          {claims.get('grant_type')}")
print(f"  framework:           {claims.get('framework')}")
print(f"  delegation_depth:    {claims.get('delegation_depth', 0)}")

# Timestamps
iat = datetime.fromtimestamp(claims['iat'], tz=timezone.utc)
exp = datetime.fromtimestamp(claims['exp'], tz=timezone.utc)
print(f"  iat (issued at):     {iat.isoformat()}")
print(f"  exp (expires at):    {exp.isoformat()}")

---

## 4. Agent-to-Agent Delegation (RFC 8693)

### The Real-World Problem

In multi-agent systems, an **orchestrator** coordinates specialized sub-agents. For example:

> *"Billing Orchestrator, process this invoice"*
> → Orchestrator delegates to **Data Fetcher** (read customer data)
> → Orchestrator delegates to **Payment Processor** (charge the card)
> → Orchestrator delegates to **Notification Agent** (send receipt)

Each sub-agent should only get the **minimum permissions it needs**, and the audit trail
should show the full chain: *who authorized what, and who delegated to whom.*

Without delegation, you'd either:
- Give every sub-agent full access (dangerous)
- Hard-code service accounts per sub-agent (brittle, no audit trail)
- Build your own token-passing system (reinventing the wheel)

### How ZeroID Handles This

ZeroID implements **RFC 8693 Token Exchange**:

1. The orchestrator holds an active token (`subject_token`)
2. The sub-agent proves its identity via a self-signed JWT assertion (`actor_token`)
3. ZeroID verifies both and issues a **delegated token** with:
   - `sub` = sub-agent's WIMSE URI (who is acting)
   - `act.sub` = orchestrator's WIMSE URI (who delegated)
   - Scopes = intersection of orchestrator's scopes, sub-agent's allowed scopes, and requested scopes

The sub-agent **cannot escalate** beyond what the orchestrator has. The delegation chain is
fully auditable. And revoking the orchestrator's token prevents future delegations.

```
Orchestrator              ZeroID                    Tool Agent
     |                       |                           |
     |-- subject_token ----->|                           |
     |                       |<--- actor_token ----------|
     |                       |    (self-signed JWT)      |
     |                       |                           |
     |                       |--- delegated_token ------>|
     |                       |    sub=tool_agent         |
     |                       |    act.sub=orchestrator   |
     |                       |    scopes=intersection    |
```

### Tool Agent Creates a Self-Signed JWT Assertion

The tool agent signs a JWT with its private key. The `iss` claim is the agent's WIMSE URI,
and the `aud` claim targets the ZeroID issuer.

In [ ]:
import jwt as pyjwt

now = int(time.time())
issuer_url = metadata["issuer"]  # from the well-known discovery response (dict)

# The tool agent builds its assertion JWT using its private key.
# This proves "I am data-fetcher" without any shared secret.
actor_assertion = pyjwt.encode(
    {
        "iss": tool_agent.wimse_uri,       # who I am (WIMSE URI)
        "aud": [issuer_url],               # target: ZeroID
        "iat": now,
        "exp": now + 300,                   # 5 minute validity
    },
    tool_agent_private_key,
    algorithm="ES256",
)

print(f"Actor assertion created:")
print(f"  Algorithm: ES256")
print(f"  Issuer:    {tool_agent.wimse_uri}")
print(f"  Audience:  {issuer_url}")
print(f"  JWT:       {actor_assertion[:60]}...")

### Token Exchange: Orchestrator Delegates to Tool Agent

In [ ]:
delegated_response = client.tokens.issue(
    grant_type="urn:ietf:params:oauth:grant-type:token-exchange",
    subject_token=orchestrator_token,    # orchestrator's active token
    actor_token=actor_assertion,          # tool agent's self-signed assertion
    scope="data:read",                    # requested scope (must be subset of both)
)

delegated_token = delegated_response.access_token

print(f"Delegated token issued:")
print(f"  Token Type:  {delegated_response.token_type}")
print(f"  Scope:       {delegated_response.scope}")
print(f"  Expires In:  {delegated_response.expires_in}s")
print(f"  Token:       {delegated_token[:60]}...")

### Inspect the Delegated Token Claims

The delegated token carries an `act` (actor) claim showing the delegation chain.
Downstream services can see both **who is acting** and **who delegated**.

In [ ]:
delegated_claims = decode_jwt_claims(delegated_token)

print("Delegated token claims:")
print(f"  sub (who is acting):       {delegated_claims.get('sub')}")
print(f"  identity_type:             {delegated_claims.get('identity_type')}")
print(f"  sub_type:                  {delegated_claims.get('sub_type')}")
print(f"  scopes:                    {delegated_claims.get('scopes')}")
print(f"  delegation_depth:          {delegated_claims.get('delegation_depth')}")
print(f"  grant_type:                {delegated_claims.get('grant_type')}")
print()

# The act claim shows who delegated authority
act = delegated_claims.get("act", {})
print(f"  act.sub (who delegated):   {act.get('sub')}")
print()

# Summary
print("Delegation chain:")
print(f"  {act.get('sub', 'unknown')}")
print(f"    └── delegated to ──> {delegated_claims.get('sub')}")
print(f"        scopes: {delegated_claims.get('scopes')}, depth: {delegated_claims.get('delegation_depth')}")

---

## 5. Token Introspection & Revocation

Tokens are bearer credentials — anyone who has one can use it. You need two capabilities:

- **Introspect** — "Is this token still valid?" Downstream services call this before granting access.
  Returns the full identity context without decoding the JWT themselves.
- **Revoke** — "Kill this token now." When an agent is compromised, misbehaving, or decommissioned,
  you revoke its credentials immediately — don't wait for expiry.

ZeroID implements RFC 7662 (introspection) and RFC 7009 (revocation).

### Introspect an Active Token

In [ ]:
introspection = client.tokens.introspect(delegated_token)

print(f"Token introspection:")
print(f"  active:          {introspection.active}")
print(f"  sub:             {introspection.sub}")
print(f"  iss:             {introspection.iss}")
print(f"  scope:           {introspection.scope}")
print(f"  trust_level:     {introspection.trust_level}")
print(f"  account_id:      {introspection.account_id}")
print(f"  project_id:      {introspection.project_id}")

# TokenIntrospection has extra="allow", so check for act via model_extra
act = getattr(introspection, "act", None) or introspection.model_extra.get("act")
if act:
    print(f"  act.sub:         {act.get('sub')}")

### Revoke the Token

In [ ]:
client.tokens.revoke(delegated_token)
print("Token revocation request sent (RFC 7009 — always returns 200).")

### Verify Revocation

In [ ]:
introspection_after = client.tokens.introspect(delegated_token)

print(f"After revocation:")
print(f"  active: {introspection_after.active}")

assert introspection_after.active is False, "Token should be inactive after revocation"
print("\nToken successfully revoked.")

---

## 6. Credential Policies

Credential policies govern token issuance. They define constraints that ZeroID enforces
before signing any JWT:

| Constraint | Description | Default |
|-----------|-------------|----------|
| `max_ttl_seconds` | Maximum token lifetime | 3600 (1 hour) |
| `max_delegation_depth` | Maximum delegation chain depth | 1 |
| `allowed_grant_types` | Which OAuth grants are permitted | `[api_key, client_credentials]` |
| `allowed_scopes` | Which scopes can be requested | (all) |
| `required_trust_level` | Minimum identity trust level | (none) |
| `required_attestation` | Minimum attestation level | (none) |

### Create a Restrictive Policy

In [ ]:
try:
    policy = client.credential_policies.create(
        name="short-lived-delegated",
        description="Short-lived tokens for delegated tool agents with depth cap",
        max_ttl_seconds=1800,                        # 30 minutes max
        max_delegation_depth=2,                       # orchestrator -> tool -> (one more)
        allowed_grant_types=[
            "client_credentials",
            "urn:ietf:params:oauth:grant-type:token-exchange",
        ],
        allowed_scopes=["data:read", "billing:read"],  # no write access
        required_trust_level="verified_third_party",    # minimum trust
    )
    print("Credential policy created (new)")
except ConflictError:
    # Already exists — fetch it by listing and filtering
    policies = client.credential_policies.list()
    policy = next(p for p in policies if p.name == "short-lived-delegated")
    print("Credential policy already exists (reusing)")

print(f"  ID:                   {policy.id}")
print(f"  Name:                 {policy.name}")
print(f"  Max TTL:              {policy.max_ttl_seconds}s ({policy.max_ttl_seconds // 60} minutes)")
print(f"  Max Delegation Depth: {policy.max_delegation_depth}")
print(f"  Allowed Grant Types:  {policy.allowed_grant_types}")
print(f"  Allowed Scopes:       {policy.allowed_scopes}")
print(f"  Required Trust Level: {policy.required_trust_level}")
print(f"  Active:               {policy.is_active}")

### List All Policies

In [ ]:
policies = client.credential_policies.list()

print(f"Total policies: {len(policies)}\n")

rows = []
for p in policies:
    rows.append([
        p.name,
        f"{p.max_ttl_seconds}s",
        str(p.max_delegation_depth),
        p.required_trust_level or "-",
        str(p.is_active),
    ])

_table(["Policy Name", "Max TTL", "Max Depth", "Trust Level", "Active"], rows)

---

## 7. CAE Signals (Real-time Revocation)

### The Problem with Token Expiry Alone

A 1-hour token means a compromised agent has up to 1 hour of unauthorized access.
Shorter TTLs help but add latency (more token refreshes). You need both:
short TTLs *and* the ability to kill tokens instantly when something goes wrong.

### How CAE Signals Work

**Continuous Access Evaluation (CAE)** lets your monitoring systems push risk signals
to ZeroID in real time. When a `high` or `critical` severity signal arrives,
ZeroID **automatically revokes all active credentials** for that identity — no human intervention.

Use cases:
- Agent accessing 500+ records in 10 seconds → `anomalous_behavior` / `critical`
- Agent's IP changed unexpectedly → `ip_change` / `high`
- Agent owner left the company → `owner_change` / `critical`
- Routine policy update → `policy_violation` / `low` (logged, not revoked)

| Signal Type | Description |
|------------|-------------|
| `credential_change` | Key material was rotated or compromised |
| `session_revoked` | Session was forcibly ended |
| `ip_change` | Unexpected IP address change |
| `anomalous_behavior` | Behavioral anomaly detected |
| `policy_violation` | Policy constraint was violated |
| `retirement` | Identity is being retired |
| `owner_change` | Identity ownership transferred |

| Severity | Auto-Revoke? |
|---------|-------------|
| `low` | No — logged only |
| `medium` | No — logged only |
| `high` | **Yes** — all active credentials revoked |
| `critical` | **Yes** — all active credentials revoked |

### Issue a Fresh Token for the Orchestrator

In [ ]:
# Get a fresh token for the orchestrator
fresh_token_response = client.tokens.issue(
    grant_type="client_credentials",
    client_id=oauth_client.client_id,
    client_secret=client_secret,
    scope="billing:read",
)
fresh_token = fresh_token_response.access_token

# Confirm it is active
result = client.tokens.introspect(fresh_token)
print(f"Fresh token active: {result.active}")
assert result.active is True

### Ingest a Critical CAE Signal

Simulate an anomalous behavior detection. The critical severity will trigger
automatic revocation of all active credentials for this identity.

In [ ]:
signal = client.signals.ingest(
    identity_id=orchestrator.id,
    signal_type="anomalous_behavior",
    severity="critical",
    source="notebook-demo",
    payload={
        "reason": "Agent accessed 500+ records in 10 seconds",
        "ip": "203.0.113.42",
        "records_accessed": 523,
    },
)

print(f"CAE signal ingested:")
print(f"  Signal ID:   {signal.id}")
print(f"  Type:        {signal.signal_type}")
print(f"  Severity:    {signal.severity}")
print(f"  Source:      {signal.source}")
print(f"  Identity ID: {signal.identity_id}")

### Verify Automatic Revocation

In [ ]:
# Poll for revocation — CAE signal triggers async revocation.
timeout_seconds = 5
start_time = time.time()
while time.time() - start_time < timeout_seconds:
    result_after_signal = client.tokens.introspect(fresh_token)
    if not result_after_signal.active:
        break
    time.sleep(0.2)

print(f"Token status after CRITICAL CAE signal:")
print(f"  active: {result_after_signal.active}")

assert result_after_signal.active is False, "Token should be auto-revoked"
print(f"\nCritical CAE signal automatically revoked all active credentials ({time.time() - start_time:.1f}s).")

### View Recent Signals

In [ ]:
recent_signals = client.signals.list(limit=5)

print(f"Recent signals ({len(recent_signals)}):")
print()

rows = []
for s in recent_signals:
    rows.append([
        s.signal_type,
        s.severity,
        s.source,
        s.identity_id[:12] + "...",
        str(s.created_at)[:19],
    ])

_table(["Type", "Severity", "Source", "Identity", "Created"], rows)

---

## 8. ZeroID Tokens + Downstream Authorization

ZeroID tokens are standard ES256 JWTs. Any service that fetches ZeroID's JWKS
(`/.well-known/jwks.json`) can verify them and use the embedded claims for authorization.

The `trust_level`, `identity_type`, `sub_type`, and `scopes` claims make it possible to
write fine-grained authorization policies. For example, using Cedar:

```cedar
// Allow first_party orchestrators to call any tool
permit (
    principal is Agent,
    action == Action::"call_tool",
    resource
)
when {
    context.trust_level == "first_party" &&
    context.identity_type == "agent" &&
    context.sub_type == "orchestrator"
};

// Deny unverified agents from sensitive operations
forbid (
    principal is Agent,
    action == Action::"call_tool",
    resource
)
when {
    context.trust_level == "unverified" &&
    resource.is_sensitive == true
};
```

Downstream services verify the JWT signature against ZeroID's JWKS, then map the claims
into their authorization context — no callback to ZeroID needed.

### Highflame Platform Integration

For teams that don't want to build their own policy engine, the
[Highflame platform](https://highflame.com) consumes ZeroID tokens natively and provides:

- **Cedar policy engine** — write and enforce fine-grained authorization policies using the ZeroID JWT claims above
- **Guardrails** — content safety, prompt injection detection, and tool call governance, all identity-aware via ZeroID
- **Observability** — trace every agent action back to its ZeroID identity, delegation chain, and trust level

ZeroID handles *who the agent is*. Highflame handles *what the agent can do*.

In [ ]:
# Get a fresh orchestrator token (previous was revoked by CAE signal)
fresh_orch_token = client.tokens.issue(
    grant_type="client_credentials",
    client_id=oauth_client.client_id,
    client_secret=client_secret,
    scope="billing:read",
).access_token

print("ZeroID JWTs carry rich identity context for downstream authorization.")
print()
print("Any service that trusts ZeroID's JWKS can verify the token and use these claims:")
print()

claims = decode_jwt_claims(fresh_orch_token)
_table(
    ["JWT Claim", "Value", "Authorization Use"],
    [
        ["sub", claims.get("sub", "")[:60] + "...", "Who is this agent?"],
        ["trust_level", claims.get("trust_level", ""), "How much do we trust it?"],
        ["identity_type", claims.get("identity_type", ""), "What kind of principal?"],
        ["sub_type", claims.get("sub_type", ""), "What role does it play?"],
        ["scopes", str(claims.get("scopes", [])), "What is it allowed to do?"],
        ["delegation_depth", str(claims.get("delegation_depth", 0)), "Is it acting on its own or delegated?"],
        ["framework", claims.get("framework", ""), "Which agent framework?"],
    ],
)

---

## Quick Reference

### SDK Client Methods

| Method | Description |
|--------|-------------|
| `client.health()` | Health check |
| `client.jwks()` | Fetch JWKS (public keys) |
| `client.identities.create(...)` | Register an identity |
| `client.identities.get(id)` | Get identity by UUID |
| `client.identities.list()` | List identities |
| `client.identities.update(id, ...)` | Update identity fields |
| `client.identities.delete(id)` | Deactivate identity (soft delete) |
| `client.oauth_clients.create(...)` | Register an OAuth2 client |
| `client.oauth_clients.list()` | List OAuth2 clients |
| `client.tokens.issue(...)` | Issue a token (all grant types) |
| `client.tokens.introspect(token)` | Introspect a token (RFC 7662) |
| `client.tokens.revoke(token)` | Revoke a token (RFC 7009) |
| `client.credential_policies.create(...)` | Create a credential policy |
| `client.credential_policies.list()` | List policies |
| `client.signals.ingest(...)` | Ingest a CAE signal |
| `client.signals.list(limit=N)` | List recent signals |

### OAuth2 Grant Types

| Grant Type | Use Case | Auth Material |
|-----------|----------|---------------|
| `client_credentials` | Machine-to-machine (RFC 6749 §4.4) | `client_id` + `client_secret` |
| `urn:ietf:params:oauth:grant-type:jwt-bearer` | Key-based (RFC 7523) | Self-signed JWT assertion |
| `urn:ietf:params:oauth:grant-type:token-exchange` | Delegation (RFC 8693) | `subject_token` + `actor_token` |
| `api_key` | SDK/CLI authentication | `zid_sk_*` API key |
| `authorization_code` | PKCE flow (CLI, MCP clients) | Auth code + code verifier |
| `refresh_token` | Token rotation | Refresh token |

### JWT Claims

| Claim | Description |
|-------|-------------|
| `iss` | Issuer (ZeroID server URL) |
| `sub` | Subject (WIMSE URI of the identity) |
| `jti` | Unique token ID |
| `iat` / `exp` | Issued at / expires at |
| `account_id` / `project_id` | Tenant scope |
| `identity_type` / `sub_type` | Identity classification |
| `trust_level` | Trust level at issuance time |
| `scopes` | Granted scopes |
| `grant_type` | How the token was issued |
| `delegation_depth` | Depth in the delegation chain (0 = direct) |
| `act.sub` | Who delegated (for token_exchange tokens) |
| `framework` / `version` / `publisher` | Agent metadata |
| `capabilities` | Agent capabilities (JSON) |

---

Built with [ZeroID](https://zeroid.dev) by [Highflame](https://highflame.com) — the control plane for autonomous AI.